# Evaluation

Today, we. are going to test our angle-based rep counter of videos of multiple reps of a certain exercise. We will measure two metrics, the average amount that our "model" is off by, and the accuracy, or how many times the model got it perfectly correct.

In this notebook, we are testing the rep counter and feedback model and it will be simple on videos of single reps strung together.

In [ ]:
import numpy as np
import cv2 as cv
import mediapipe as mp
import random
import os
from tqdm import tqdm
import tensorflow as tf
from collections import Counter

In [ ]:
@tf.keras.utils.register_keras_serializable()
class Lunge(tf.keras.Model):
    def __init__(self, num_heads=4, key_dim=32, **kwargs):
        super().__init__(**kwargs)

        self.conv1 = tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same")
        self.pool1 = tf.keras.layers.MaxPooling1D()

        self.conv2 = tf.keras.layers.Conv1D(64, 3, activation="relu", padding="same")
        self.pool2 = tf.keras.layers.MaxPooling1D()

        self.mha1 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm1 = tf.keras.layers.LayerNormalization()

        self.mha2 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm2 = tf.keras.layers.LayerNormalization()

        self.bn = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling1D()

        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.dense2 = tf.keras.layers.Dense(256, activation="relu")
        self.out = tf.keras.layers.Dense(3, activation="softmax")

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        attn1 = self.mha1(x, x, training=training)
        x = self.norm1(x + attn1)

        attn2 = self.mha2(x, x, training=training)
        x = self.norm2(x + attn2)

        x = self.bn(x, training=training)

        x = self.gap(x)

        x = self.dense1(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)

        return self.out(x)


In [ ]:
@tf.keras.utils.register_keras_serializable()
class Pushup(tf.keras.Model):
    def __init__(self, num_heads=4, key_dim=32, **kwargs):
        super().__init__(**kwargs)

        self.conv1 = tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same")
        self.pool1 = tf.keras.layers.MaxPooling1D()

        self.conv2 = tf.keras.layers.Conv1D(64, 3, activation="relu", padding="same")
        self.pool2 = tf.keras.layers.MaxPooling1D()

        self.mha1 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm1 = tf.keras.layers.LayerNormalization()

        self.mha2 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm2 = tf.keras.layers.LayerNormalization()

        self.bn = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling1D()

        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.dense2 = tf.keras.layers.Dense(256, activation="relu")
        self.out = tf.keras.layers.Dense(3, activation="softmax")

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        attn1 = self.mha1(x, x, training=training)
        x = self.norm1(x + attn1)

        attn2 = self.mha2(x, x, training=training)
        x = self.norm2(x + attn2)

        x = self.bn(x, training=training)

        x = self.gap(x)

        x = self.dense1(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)

        return self.out(x)
    

In [ ]:
@tf.keras.utils.register_keras_serializable()
class Squat(tf.keras.Model):
    def __init__(self, num_heads=4, key_dim=32, **kwargs):
        super().__init__(**kwargs)

        self.conv1 = tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same")
        self.pool1 = tf.keras.layers.MaxPooling1D()

        self.conv2 = tf.keras.layers.Conv1D(64, 3, activation="relu", padding="same")
        self.pool2 = tf.keras.layers.MaxPooling1D()

        self.mha1 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm1 = tf.keras.layers.LayerNormalization()

        self.mha2 = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm2 = tf.keras.layers.LayerNormalization()

        self.bn = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling1D()

        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.dense2 = tf.keras.layers.Dense(256, activation="relu")
        self.out = tf.keras.layers.Dense(1, activation="sigmoid")

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        attn1 = self.mha1(x, x, training=training)
        x = self.norm1(x + attn1)

        attn2 = self.mha2(x, x, training=training)
        x = self.norm2(x + attn2)

        x = self.bn(x, training=training)

        x = self.gap(x)

        x = self.dense1(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)

        return self.out(x)
    

In [ ]:
# same repcounter that is in src/rep_counter.py

class RepCounter:

    def __init__(self, exercise: str):
        self.exercise = exercise
        BaseOptions = mp.tasks.BaseOptions
        self.PoseLandmarker = mp.tasks.vision.PoseLandmarker
        PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
        VisionRunningMode = mp.tasks.vision.RunningMode

        self.options = PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path='/Users/ansh/Downloads/development/repquest/pose_landmarker_full.task'), # this has been modified
            running_mode=VisionRunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            output_segmentation_masks=False,
        )

        self.rep_map = {
            "pushup": (60, 135),
            "squat": (87.5, 160),
            "lunge": (90, 140),
        }

        self.rep_map_kypts = {
            "pushup": [[15, 13, 11], [16, 14, 12]], # right points, left points
            "squat": [[23, 25, 27], [24, 26, 28]],
            "lunge": [[23, 25, 27], [24, 26, 28]],
        }

        if self.exercise not in self.rep_map.keys():
            raise ValueError("enter valid exercise")
        
        self.MIN_ANGLE, self.UPRIGHT_POS_ANGLE = self.rep_map[self.exercise]
        self.WAIT_FRAMES = 20

    def convert_landmarks(self, pose):
        data = []
        for landmark_list in pose.pose_landmarks:
            landmarks_array = np.array([
                [lm.x, lm.y, lm.z] for lm in landmark_list
            ])
            data.append(landmarks_array)
        return np.array(data)

    def calculate_angle(self, a, b, c):  # was missing `self`
        ba = a - b
        bc = c - b
        cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos_a, -1, 1)))  # was np.clip(-1, 1)

    def count_reps(self, path, show_img: bool, return_pose=False):
        cap = cv.VideoCapture(path) # can be 0 or an actual path
        
        # cycle steps
        self.initial = False
        self.low = False
        self.back_up = False
        self.wait_frames_remaining = 0
        self.wait_over = True
        self.n_reps = 0
        frame_idx = 0

        pose = []

        with self.PoseLandmarker.create_from_options(self.options) as self.landmarker:
            while True:
                ret, frame = cap.read()
                if not ret: break
                frame = cv.flip(frame, 1)
                rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB) # convert to rgb
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame) # convert to mp image
                timestamp_ms = int((frame_idx/30) * 1000) # get timestamp
                result = self.landmarker.detect_for_video(mp_img, timestamp_ms) # inference
                if not result.pose_landmarks:
                    frame_idx += 1
                    if show_img:
                        cv.imshow("frame", frame)
                        continue
                    continue
                landmark_arr = self.convert_landmarks(pose=result)
                
                if return_pose:
                    pose.append(landmark_arr)
                
                kypts = self.rep_map_kypts[self.exercise]
                right_pt1 = landmark_arr[0, kypts[0][0]]
                right_pt2 = landmark_arr[0, kypts[0][1]]
                right_pt3 = landmark_arr[0, kypts[0][2]]
                left_pt1  = landmark_arr[0, kypts[1][0]]
                left_pt2  = landmark_arr[0, kypts[1][1]]
                left_pt3  = landmark_arr[0, kypts[1][2]]
                
                right_angle = self.calculate_angle(right_pt1, right_pt2, right_pt3)
                left_angle = self.calculate_angle(left_pt1, left_pt2, left_pt3)

                if self.wait_over:
                    if not self.initial and not self.low and not self.back_up:
                        if right_angle > self.UPRIGHT_POS_ANGLE or left_angle > self.UPRIGHT_POS_ANGLE:
                            self.initial = True
                    elif self.initial and not self.low:
                        if right_angle < self.MIN_ANGLE or left_angle < self.MIN_ANGLE:
                            self.low = True
                    elif self.initial and self.low and not self.back_up:
                        if right_angle > self.UPRIGHT_POS_ANGLE or left_angle > self.UPRIGHT_POS_ANGLE:
                            self.back_up = True
                    if self.initial and self.low and self.back_up:
                        self.initial, self.low, self.back_up = False, False, False
                        self.wait_over = False
                        self.wait_frames_remaining = self.WAIT_FRAMES
                        self.n_reps += 1
                
                else:
                    self.wait_frames_remaining -= 1
                    if self.wait_frames_remaining == 0:
                        self.wait_over = True
                frame_idx += 1


                if show_img:
                    cv.imshow("frame", frame)
                if cv.waitKey(1) & 0xFF == ord('q'):
                    break
        
        if not return_pose:
            return self.n_reps
        else:
            return self.n_reps, pose

In [ ]:
# universal class that handles: data preprocessing and training

class Exercise:

    def __init__(self, exercise_name: str, model, on_colab):
        self.exercise_name = exercise_name
        self.on_colab = on_colab

        # task config
        self.BaseOptions = mp.tasks.BaseOptions
        self.PoseLandmarker = mp.tasks.vision.PoseLandmarker
        self.PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
        self.VisionRunningMode = mp.tasks.vision.RunningMode

        self.options = self.PoseLandmarkerOptions(
            base_options=self.BaseOptions(model_asset_path='repquest/pose_landmarker_full.task' if self.on_colab else 'pose_landmarker_full.task'),
            running_mode=self.VisionRunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            output_segmentation_masks=False,
        )

        self.fps = 30
        self.model = model

    def landmarks_to_numpy(self, result, include_visibility=False):

        poses = []
        for pose_landmarks in result.pose_landmarks:
            if include_visibility:
                arr = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in pose_landmarks])
            else:
                arr = np.array([[lm.x, lm.y, lm.z] for lm in pose_landmarks])
            poses.append(arr)
        
        return np.array(poses)

    def angle(self, a, b, c):
        ba = a - b
        bc = c - b
        cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

    def midpoint(self, a, b):
        return (a + b) / 2
    
    def augment_pose(self, pose_arr):
        
        if pose_arr.shape[0] == 0:  # no pose detected, return as-is
            return pose_arr

        p = pose_arr[0].copy()

        # gaussian noise
        p += np.random.normal(0, 0.01, p.shape)

        # scale jitter (around the pose center)
        scale = np.random.uniform(0.9, 1.1)
        center = p.mean(axis=0)
        p = (p - center) * scale + center

        # translation jitter
        shift = np.random.uniform(-0.05, 0.05, size=(1, 3))
        shift[0, 2] = 0  # don't shift z
        p += shift

        # torizontal flip (50% chance)
        if np.random.rand() < 0.5:
            p[:, 0] = 1 - p[:, 0]  # flip x coordinates

            # swap left/right landmark pairs
            FLIP_PAIRS = [
                (11, 12), (13, 14), (15, 16),  # shoulders, elbows, wrists
                (17, 18), (19, 20), (21, 22),  # hands
                (23, 24), (25, 26), (27, 28),  # hips, knees, ankles
                (29, 30), (31, 32),            # heels, foot index
            ]
            for l, r in FLIP_PAIRS:
                p[l], p[r] = p[r].copy(), p[l].copy()

        return p[np.newaxis, :]  # restore (1, 33, 3)
    
    def extract_features(self, pose_arr):
    
        empty = np.zeros(11 + 99)

        if pose_arr.shape[0] == 0:  # no pose detected
            return empty

        p = pose_arr[0]  # shape (33, 3)

        l_shoulder = p[11, :2]
        r_shoulder = p[12, :2]
        l_elbow = p[13, :2]
        r_elbow = p[14, :2]
        l_wrist = p[15, :2]
        r_wrist = p[16, :2]
        l_hip = p[23, :2]
        r_hip = p[24, :2]
        l_knee = p[25, :2]
        r_knee = p[26, :2]
        l_ankle = p[27, :2]
        r_ankle = p[28, :2]

        elbow_angle_l = self.angle(l_shoulder, l_elbow, l_wrist)
        elbow_angle_r = self.angle(r_shoulder, r_elbow, r_wrist)
        elbow_angle = (elbow_angle_l + elbow_angle_r) / 2

        mid_shoulder = self.midpoint(l_shoulder, r_shoulder)
        mid_hip = self.midpoint(l_hip, r_hip)
        mid_knee = self.midpoint(l_knee, r_knee)
        mid_ankle = self.midpoint(l_ankle, r_ankle)
        body_line_angle = self.angle(mid_shoulder, mid_hip, mid_knee)

        shoulder_ankle_vec = mid_ankle - mid_shoulder
        shoulder_hip_vec = mid_hip - mid_shoulder
        line_len = np.linalg.norm(shoulder_ankle_vec) + 1e-6
        t = np.dot(shoulder_hip_vec, shoulder_ankle_vec) / (line_len ** 2)
        projected = mid_shoulder + t * shoulder_ankle_vec
        hip_offset = mid_hip[1] - projected[1]

        shoulder_angle_l = self.angle(l_elbow, l_shoulder, l_hip)
        shoulder_angle_r = self.angle(r_elbow, r_shoulder, r_hip)
        shoulder_angle = (shoulder_angle_l + shoulder_angle_r) / 2

        wrist_shoulder_dist = (np.linalg.norm(l_wrist - l_shoulder) + np.linalg.norm(r_wrist - r_shoulder)) / 2
        hip_shoulder_y_offset = mid_hip[1] - mid_shoulder[1]

        knee_angle_l = self.angle(l_hip, l_knee, l_ankle)
        knee_angle_r = self.angle(r_hip, r_knee, r_ankle)
        knee_angle = (knee_angle_l + knee_angle_r) / 2

        features = np.array([
            elbow_angle,
            body_line_angle,
            hip_offset,
            shoulder_angle,
            wrist_shoulder_dist,
            hip_shoulder_y_offset,
            knee_angle,
            elbow_angle_l,
            elbow_angle_r,
            knee_angle_l,
            knee_angle_r,
        ], dtype=np.float32)

        landmarks_flat = p.flatten()

        combined = np.concatenate([features, landmarks_flat])

        return combined
    
    def data_preprocess(self, n_augments):

        data_path = f"repquest/video_data/{self.exercise_name}" if self.on_colab else f"video_data/{self.exercise_name}"
        folders = sorted(os.listdir(data_path))
        folders = [folder for folder in folders if folder != ".DS_Store"]
        self.n_classes = len(folders)
        self.n_augments = n_augments

        X_train = []
        y_train = []
        
        with self.PoseLandmarker.create_from_options(self.options) as landmarker:

            frame_idx = 0
        
            for class_label in tqdm(range(self.n_classes), desc="processing classes"):
            
                curr_folder = folders[class_label]
                
                videos = os.listdir(f"{data_path}/{curr_folder}")
                videos = [video for video in videos if video != ".DS_Store"]

                for video in tqdm(videos, desc=f"  {curr_folder}", leave=False):

                    cap = cv.VideoCapture(f"{data_path}/{curr_folder}/{video}")

                    buffer = []
                    pose_buffer = []
                    
                    while True:
                        ret, frame = cap.read()
                        if not ret: break
                        frame = cv.resize(frame, (640, 480))
                        rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
                        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                        timestamp_ms = int((frame_idx / self.fps) * 1000)
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        pose_arr = self.landmarks_to_numpy(result)
                        pose_buffer.append(pose_arr)
                        derived_features = self.extract_features(pose_arr)
                        buffer.append(derived_features)
                        frame_idx += 1

                    cap.release()

                    for _ in range(self.n_augments):
                        aug_buffer = []
                        for pose in pose_buffer:
                            pose = self.augment_pose(pose)
                            features = self.extract_features(pose)
                            aug_buffer.append(features)
                        X_train.append(np.array(aug_buffer))
                        y_train.append(class_label)

                    X_train.append(np.array(buffer))
                    y_train.append(class_label)

            X_train = np.array(X_train)
            y_train = np.array(y_train)

            idx = np.random.permutation(len(X_train))
            X_train, y_train = X_train[idx], y_train[idx]

            save_path = f"repquest/data/{self.exercise_name}" if self.on_colab else f"data/{self.exercise_name}"
            os.makedirs(save_path, exist_ok=True)
            np.save(f"{save_path}/X.npy", X_train)
            np.save(f"{save_path}/y.npy", y_train)

        return folders
    
    def train(self, optimizer, epochs, batch_size, shuffle, data_dir, loss):

        # model checkpoint
        model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
            f'drive/MyDrive/colab_checkpoints/{self.exercise_name}.keras' if self.on_colab else f'models/{self.exercise_name}.keras',
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        )

        # early stopping 
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True
        )

        self.callbacks = [model_checkpoint, early_stopping]
        
        self.optimizer = optimizer
        self.epochs = epochs
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.loss = loss

        # load data
        X = np.load(f"{data_dir}/X.npy")
        y = np.load(f"{data_dir}/y.npy")

        self.model.compile(
            self.optimizer,
            self.loss,
            metrics=['accuracy']
        )

        self.model.fit(
            X,
            y,
            epochs=self.epochs,
            shuffle=self.shuffle,
            validation_split=0.15,
            batch_size=self.batch_size,
            callbacks=self.callbacks,
        )

In [ ]:
def create_sample_and_test(exercise: str):

    GOOD_REP_WEIGHT = 3

    n_reps = random.randint(1, 20)
    n_counted_reps = 0
    workout_labels = []

    ROOT_DIR = f"/Users/ansh/Downloads/development/repquest/video_data/{exercise}"
    variations = os.listdir(ROOT_DIR)
    all_filenames = []
    for directory in variations:
        if directory == ".DS_Store":
            continue
        files = os.listdir(f"{ROOT_DIR}/{directory}")
        for file in files:
            if file != ".DS_Store":
                all_filenames.append(f"{directory}/{file}")
                workout_labels.append(directory)
    
    weights = [GOOD_REP_WEIGHT if "good" in f else 1 for f in all_filenames]

    os.makedirs("/Users/ansh/Downloads/development/repquest/temp_videos/", exist_ok=True)
    out = cv.VideoWriter(
        "/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4",
        cv.VideoWriter_fourcc(*"mp4v"),
        30,
        (1920, 1080),
    )

    for rep in range(n_reps):
        vid = random.choices(all_filenames, weights=weights, k=1)[0]
        if "good" in vid:
            n_counted_reps += 1
        
        cap = cv.VideoCapture(f"{ROOT_DIR}/{vid}")
        while True:
            ret, frame = cap.read()
            if not ret: break
            out.write(frame)
        cap.release()

    out.release()

    rep_counter = RepCounter(exercise=exercise)
    predicted_reps = rep_counter.count_reps(
        path="/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4",
        show_img=False,
        return_pose=False,
    )

    buffer = []
    SEQ_LEN = 106
    predicted_forms = []
    frame_idx = 0
    model, label_map = None, None

    if exercise == "pushup":
        model = tf.keras.models.load_model(
            "/Users/ansh/Downloads/development/repquest/models/pushup.keras",
            custom_objects={"Pushup": Pushup}
        )
        label_map = {
            "good_pushup": 0,
            "high_hip_pushup": 1,
            "low_hip_pushup": 2,
        }
    elif exercise == "squat":
        model = tf.keras.models.load_model(
            "/Users/ansh/Downloads/development/repquest/models/squat.keras",
            custom_objects={"Squat": Squat}
        )
        label_map = {
            "good_squat": 0,
            "partial_squat": 1,
        }
    else:
        model = tf.keras.models.load_model(
            "/Users/ansh/Downloads/development/repquest/models/lunge.keras",
            custom_objects={"Lunge": Lunge}
        )
        label_map = {
            "angled_back_lunge": 0,
            "good_lunge": 1,
            "partial_lunge": 2,
        }

    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='/Users/ansh/Downloads/development/repquest/pose_landmarker_full.task'), # this has been modified
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    exercise_util_class = Exercise(exercise_name=exercise, model=None, on_colab=False)

    cap = cv.VideoCapture("/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4")
    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv.flip(frame, 1)
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp = int((frame_idx/30)*1000)
            result = landmarker.detect_for_video(mp_img, timestamp)
            if not result.pose_landmarks:
                frame_idx += 1
                continue
            else:
                landmark_arr = exercise_util_class.landmarks_to_numpy(result)
                landmark_arr = exercise_util_class.extract_features(landmark_arr)
                buffer.append(landmark_arr)
            
            if len(buffer) == SEQ_LEN:
                input_tensor = tf.stack(buffer, axis=0)
                input_tensor = tf.expand_dims(input_tensor, axis=0)
                pred = model(input_tensor, training=False)
                pred_class = tf.argmax(pred, axis=-1).numpy()[0]
                current_prediction = list(label_map.keys())[pred_class]
                buffer = []
                predicted_forms.append(current_prediction)

            frame_idx += 1

    os.remove("/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4")

    return {
        "actual": n_counted_reps,
        "predicted": predicted_reps,
        "actual_form": workout_labels,
        "predicted_form": predicted_forms
    }

In [ ]:
import csv
import os
import logging
from tqdm import tqdm

# suppress mediapipe logs
os.environ["GLOG_minloglevel"] = "3"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("mediapipe").setLevel(logging.ERROR)

def dominant_form(forms: list[str]) -> str:
    bad_forms = [f for f in forms if "good" not in f]
    if bad_forms:
        return Counter(bad_forms).most_common(1)[0][0]
    return Counter(forms).most_common(1)[0][0]

n = 300
correct = 0
incorrect = 0
diff = 0
form_correct = 0
form_total = 0

csv_filename = "model_results.csv"

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "iteration",
            "exercise",
            "actual reps",
            "predicted reps",
            "dominant actual form",
            "dominant predicted form",
            "rep counting correct",
            "form classification correct",
        ]
    )
    f.flush()

    for i in tqdm(range(n), desc="testing"):
        exercise = random.choice(["pushup", "squat", "lunge"])
        out = create_sample_and_test(exercise=exercise)

        actual = out["actual"]
        prediction = out["predicted"]
        actual_forms = out["actual_form"]
        predicted_forms = out["predicted_form"]

        is_rep_correct = actual == prediction
        if is_rep_correct:
            correct += 1
        else:
            incorrect += 1
            diff += abs(actual - prediction)

        dominant_actual = "none"
        dominant_predicted = "none"
        is_form_correct = False

        if predicted_forms:
            form_total += 1
            dominant_actual = dominant_form(actual_forms)
            dominant_predicted = dominant_form(predicted_forms)
            if dominant_actual == dominant_predicted:
                form_correct += 1
                is_form_correct = True

        writer.writerow(
            [
                i + 1,
                exercise,
                actual,
                prediction,
                dominant_actual,
                dominant_predicted,
                int(is_rep_correct),
                int(is_form_correct),
            ]
        )
        f.flush()

        rep_acc = correct / (i + 1)
        form_acc = form_correct / form_total if form_total > 0 else 0
        tqdm.write(
            f"[{i+1}/{n}] {exercise} | "
            f"actual: {actual}, predicted: {prediction} | "
            f"form: {dominant_actual} → {dominant_predicted} | "
            f"rep_acc: {rep_acc:.0%} | form_acc: {form_acc:.0%}"
        )

avg_diff = diff / incorrect if incorrect > 0 else 0
print(f"\n--- final results ---")
print(
    f"rep counting:  {correct}/{n} ({correct/n:.0%}) | avg diff when wrong: {avg_diff:.2f}"
)
print(f"form accuracy: {form_correct}/{form_total} ({form_correct/form_total:.0%} of evaluated)")
print(f"all data successfully logged into: {csv_filename}")